# NYC RL Pipeline

## Dependancies and Data Loading

### Dependancies

In [1]:
# ============================================================
# CELL 1 — Install dependencies
# ============================================================
# We pin versions to avoid Colab's default conflicts.
# torch_geometric is NOT used — we implement sparse GCN manually.

!pip install -q \
    gymnasium==0.29.1 \
    polars==0.20.31 \
    pyarrow==15.0.2 \
    geopandas==0.14.4 \
    shapely==2.0.4 \
    duckdb==0.10.3 \
    requests==2.31.0

print("✓ Packages installed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 30.1 MB/s eta 0:00:0000:01
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.
✓ Packages installed


### GPU and Global seeds

In [2]:
# ============================================================
# CELL 2 — Verify GPU + set global seeds
# ============================================================
import torch
import numpy as np
import random
import os

def set_seeds(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seeds(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("  ⚠ No GPU detected — go to Runtime > Change runtime type > T4 GPU")

✓ Device: cuda
  GPU: Tesla T4
  VRAM: 15.6 GB


### COnstants

In [3]:
# ============================================================
# CELL 3 — Global constants (single source of truth)
# ============================================================

# --- Data ---
TLC_URL = (
    "https://d37ci6vzurychx.cloudfront.net/trip-data/"
    "yellow_tripdata_2023-01.parquet"
)
ZONE_SHAPEFILE_URL = (
    "https://d37ci6vzurychx.cloudfront.net/misc/"
    "taxi_zones.zip"
)
DATA_DIR   = "/content/tlc_data"
RAW_FILE   = f"{DATA_DIR}/yellow_2023_01.parquet"
ZONE_ZIP   = f"{DATA_DIR}/taxi_zones.zip"
ZONE_DIR   = f"{DATA_DIR}/taxi_zones"

# --- Zone graph ---
N_ZONES     = 263          # valid TLC zone IDs: 1–263
MAX_DEGREE  = 10           # padded neighbour dimension (NYC zones max ~8)

# --- Time binning ---
BIN_MINUTES = 30           # 30-min timesteps  → 48 bins/day
BINS_PER_DAY = 24 * 60 // BIN_MINUTES   # 48
N_DAYS_TRAIN = 24          # first 24 days of Jan  → train
N_DAYS_TEST  =  7          # last 7 days of Jan    → eval
N_BINS_TRAIN = N_DAYS_TRAIN * BINS_PER_DAY   # 1152
N_BINS_TEST  = N_DAYS_TEST  * BINS_PER_DAY   # 336

# --- Environment ---
N_DRIVERS   = 3000         # total fleet size (realistic for NYC yellow cab)
DEMAND_FORECAST_HORIZON = 3  # look-ahead bins for projected demand feature

# --- Reward weights (will be z-score normalised at runtime) ---
ALPHA = 1.0   # revenue weight
BETA  = 0.3   # relocation cost weight
GAMMA = 0.5   # unmet demand penalty weight

# --- MAPPO training ---
N_EPISODES     = 600
ROLLOUT_STEPS  = 48        # one full simulated day per rollout
GAMMA_RL       = 0.99      # discount factor
GAE_LAMBDA     = 0.95
CLIP_EPS       = 0.2
ENTROPY_COEF   = 0.01
VALUE_LOSS_COEF= 0.5
LR_ACTOR       = 3e-4
LR_CRITIC      = 1e-3
BATCH_SIZE     = 256
PPO_EPOCHS     = 4
GRAD_CLIP      = 0.5
REWARD_CLIP    = 10.0

# --- GCN architecture ---
GCN_IN_DIM   = 7           # state features per zone (defined in Phase 1d)
GCN_HIDDEN   = 64
GCN_OUT_DIM  = 32

os.makedirs(DATA_DIR, exist_ok=True)
print("✓ Constants defined")
print(f"  Zones        : {N_ZONES}")
print(f"  Timestep     : {BIN_MINUTES} min  →  {BINS_PER_DAY} bins/day")
print(f"  Fleet size   : {N_DRIVERS} drivers")
print(f"  Train bins   : {N_BINS_TRAIN}  ({N_DAYS_TRAIN} days)")
print(f"  Test bins    : {N_BINS_TEST}  ({N_DAYS_TEST} days)")
print(f"  PPO episodes : {N_EPISODES}")

✓ Constants defined
  Zones        : 263
  Timestep     : 30 min  →  48 bins/day
  Fleet size   : 3000 drivers
  Train bins   : 1152  (24 days)
  Test bins    : 336  (7 days)
  PPO episodes : 600


### Download raw data

In [4]:
# ============================================================
# CELL 4 — Download raw data
# ============================================================
import requests
import zipfile
from pathlib import Path

def download_file(url: str, dest: str, desc: str) -> None:
    dest_path = Path(dest)
    if dest_path.exists():
        print(f"  ↳ {desc} already exists, skipping download")
        return
    print(f"  ↳ Downloading {desc} ...")
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        downloaded = 0
        with open(dest, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):  # 1 MB chunks
                f.write(chunk)
                downloaded += len(chunk)
                if total:
                    print(f"\r    {downloaded/1e6:.1f} / {total/1e6:.1f} MB", end="")
    print(f"\n  ✓ Saved → {dest}")

# Yellow taxi parquet (~500 MB)
download_file(TLC_URL, RAW_FILE, "Yellow Taxi Jan-2023")

# Zone shapefile zip (~1 MB)
download_file(ZONE_SHAPEFILE_URL, ZONE_ZIP, "Taxi Zone Shapefile")

# Unzip shapefile
if not Path(ZONE_DIR).exists():
    with zipfile.ZipFile(ZONE_ZIP, "r") as z:
        z.extractall(ZONE_DIR)
    print(f"  ✓ Shapefile extracted → {ZONE_DIR}")
else:
    print(f"  ↳ Shapefile already extracted")

  ↳ Yellow Taxi Jan-2023 already exists, skipping download
  ↳ Taxi Zone Shapefile already exists, skipping download
  ↳ Shapefile already extracted


### Load

In [5]:
# ============================================================
# CELL 5 — Load & filter raw parquet with DuckDB
# ============================================================
import duckdb
import polars as pl

print("Loading parquet via DuckDB ...")

# DuckDB reads parquet lazily — no full RAM load
con = duckdb.connect()

raw_schema = con.execute(f"DESCRIBE SELECT * FROM '{RAW_FILE}' LIMIT 1").df()
print("\nRaw schema:")
print(raw_schema[["column_name", "column_type"]].to_string(index=False))

Loading parquet via DuckDB ...

Raw schema:
          column_name column_type
             VendorID      BIGINT
 tpep_pickup_datetime   TIMESTAMP
tpep_dropoff_datetime   TIMESTAMP
      passenger_count      DOUBLE
        trip_distance      DOUBLE
           RatecodeID      DOUBLE
   store_and_fwd_flag     VARCHAR
         PULocationID      BIGINT
         DOLocationID      BIGINT
         payment_type      BIGINT
          fare_amount      DOUBLE
                extra      DOUBLE
              mta_tax      DOUBLE
           tip_amount      DOUBLE
         tolls_amount      DOUBLE
improvement_surcharge      DOUBLE
         total_amount      DOUBLE
 congestion_surcharge      DOUBLE
          airport_fee      DOUBLE


### FIltering

In [6]:
# ============================================================
# CELL 6 — Filter, clean and bin trips
# ============================================================

QUERY = f"""
SELECT
    tpep_pickup_datetime                        AS pickup_ts,
    tpep_dropoff_datetime                       AS dropoff_ts,
    PULocationID                                AS pu_zone,
    DOLocationID                                AS do_zone,
    CAST(fare_amount + tip_amount AS DOUBLE)    AS revenue,

    -- travel time in minutes
    DATEDIFF('minute', tpep_pickup_datetime,
                       tpep_dropoff_datetime)   AS travel_minutes,

    -- 30-min bin index within the month
    -- bin 0 = first 30 min of 2023-01-01 00:00
    FLOOR(
        DATEDIFF('minute',
                 TIMESTAMP '2023-01-01 00:00:00',
                 tpep_pickup_datetime)
        / {BIN_MINUTES}
    )::INTEGER AS time_bin

FROM '{RAW_FILE}'

WHERE
    -- keep only Jan 2023 trips
    tpep_pickup_datetime  >= TIMESTAMP '2023-01-01 00:00:00'
    AND tpep_pickup_datetime  <  TIMESTAMP '2023-02-01 00:00:00'
    AND tpep_dropoff_datetime >= tpep_pickup_datetime

    -- valid zone IDs only (1–263)
    AND pu_zone BETWEEN 1 AND {N_ZONES}
    AND do_zone BETWEEN 1 AND {N_ZONES}

    -- sanity: revenue > 0, travel time 1–120 min
    AND fare_amount  > 0
    AND revenue      > 0
    AND travel_minutes BETWEEN 1 AND 120

    -- sanity: passenger count
    AND passenger_count >= 1
"""

print("Running DuckDB query ...")
trips_df: pl.DataFrame = pl.from_arrow(con.execute(QUERY).arrow())
print(f"✓ Filtered trips: {len(trips_df):,}")
print(f"  Time bins present: {trips_df['time_bin'].min()} – {trips_df['time_bin'].max()}")
print(f"  Revenue range   : ${trips_df['revenue'].min():.2f} – ${trips_df['revenue'].max():.2f}")
print(f"  Travel time (min): {trips_df['travel_minutes'].min()} – {trips_df['travel_minutes'].max()}")
print(f"\nSample rows:")
print(trips_df.head(5))

Running DuckDB query ...
✓ Filtered trips: 2,843,292
  Time bins present: 0 – 1487
  Revenue range   : $0.01 – $999.00
  Travel time (min): 1 – 120

Sample rows:
shape: (5, 7)
┌──────────────┬─────────────────────┬─────────┬─────────┬─────────┬────────────────┬──────────┐
│ pickup_ts    ┆ dropoff_ts          ┆ pu_zone ┆ do_zone ┆ revenue ┆ travel_minutes ┆ time_bin │
│ ---          ┆ ---                 ┆ ---     ┆ ---     ┆ ---     ┆ ---            ┆ ---      │
│ datetime[μs] ┆ datetime[μs]        ┆ i64     ┆ i64     ┆ f64     ┆ i64            ┆ i32      │
╞══════════════╪═════════════════════╪═════════╪═════════╪═════════╪════════════════╪══════════╡
│ 2023-01-01   ┆ 2023-01-01 00:40:36 ┆ 161     ┆ 141     ┆ 9.3     ┆ 8              ┆ 1        │
│ 00:32:10     ┆                     ┆         ┆         ┆         ┆                ┆          │
│ 2023-01-01   ┆ 2023-01-01 01:01:27 ┆ 43      ┆ 237     ┆ 11.9    ┆ 6              ┆ 1        │
│ 00:55:08     ┆                     ┆         ┆

In [7]:
# ============================================================
# CELL 7 — Build all core matrices from trips_df
# ============================================================

T_TOTAL = 31 * BINS_PER_DAY   # 1488 bins total in January

# ---- 1. DEMAND MATRIX [T, N] ----
demand_raw = (
    trips_df
    .group_by(["time_bin", "pu_zone"])
    .agg(pl.len().alias("pickups"))
    .with_columns((pl.col("pu_zone") - 1).alias("zone_idx"))
)

demand_matrix = np.zeros((T_TOTAL, N_ZONES), dtype=np.float32)
for row in demand_raw.iter_rows(named=True):
    t = int(row["time_bin"])
    z = int(row["zone_idx"])
    if 0 <= t < T_TOTAL and 0 <= z < N_ZONES:
        demand_matrix[t, z] = row["pickups"]

print(f"✓ Demand matrix: {demand_matrix.shape}")
print(f"  Total pickups : {demand_matrix.sum():,.0f}")
print(f"  Max/bin/zone  : {demand_matrix.max():.0f}")
print(f"  Sparsity      : {(demand_matrix == 0).mean()*100:.1f}% zero")

# ---- 2. TRAVEL-TIME MATRIX [N, N] ----
travel_raw = (
    trips_df
    .group_by(["pu_zone", "do_zone"])
    .agg(pl.median("travel_minutes").alias("med_travel"))
    .with_columns([
        (pl.col("pu_zone") - 1).alias("pu_idx"),
        (pl.col("do_zone") - 1).alias("do_idx"),
    ])
)

travel_matrix = np.full((N_ZONES, N_ZONES), np.nan, dtype=np.float32)
np.fill_diagonal(travel_matrix, 0.0)

for row in travel_raw.iter_rows(named=True):
    i = int(row["pu_idx"])
    j = int(row["do_idx"])
    if 0 <= i < N_ZONES and 0 <= j < N_ZONES:
        travel_matrix[i, j] = row["med_travel"]

# Fill missing pairs via symmetry + global median
for i in range(N_ZONES):
    for j in range(N_ZONES):
        if np.isnan(travel_matrix[i, j]) and not np.isnan(travel_matrix[j, i]):
            travel_matrix[i, j] = travel_matrix[j, i]

global_med = np.nanmedian(travel_matrix)
travel_matrix = np.where(np.isnan(travel_matrix), global_med, travel_matrix)
travel_bins = np.ceil(travel_matrix / BIN_MINUTES).astype(np.int32)
travel_bins = np.clip(travel_bins, 1, 10)

print(f"✓ Travel matrix: {travel_matrix.shape}")
print(f"  Travel range (min) : {travel_matrix.min():.1f} – {travel_matrix.max():.1f}")
print(f"  Travel bins range  : {travel_bins.min()} – {travel_bins.max()}")

# ---- 3. REVENUE MATRIX [N, N] ----
revenue_raw = (
    trips_df
    .group_by(["pu_zone", "do_zone"])
    .agg(pl.mean("revenue").alias("avg_rev"))
    .with_columns([
        (pl.col("pu_zone") - 1).alias("pu_idx"),
        (pl.col("do_zone") - 1).alias("do_idx"),
    ])
)

avg_revenue = np.full((N_ZONES, N_ZONES), 0.0, dtype=np.float32)
for row in revenue_raw.iter_rows(named=True):
    i = int(row["pu_idx"])
    j = int(row["do_idx"])
    if 0 <= i < N_ZONES and 0 <= j < N_ZONES:
        avg_revenue[i, j] = row["avg_rev"]

global_avg_rev = avg_revenue[avg_revenue > 0].mean()
avg_revenue = np.where(avg_revenue == 0, global_avg_rev, avg_revenue)

print(f"✓ Revenue matrix: {avg_revenue.shape}")
print(f"  Revenue range  : ${avg_revenue.min():.2f} – ${avg_revenue.max():.2f}")

# ---- 4. PROJECTED DEMAND [T, N] ----
H = DEMAND_FORECAST_HORIZON
projected_demand = np.zeros((T_TOTAL, N_ZONES), dtype=np.float32)
for h in range(1, H + 1):
    shifted = np.roll(demand_matrix, -h, axis=0)
    shifted[T_TOTAL - h:] = 0.0
    projected_demand += shifted
projected_demand /= H

print(f"✓ Projected demand: {projected_demand.shape}")

# ---- 5. NORMALISE DEMAND ----
demand_max     = demand_matrix.max(axis=0, keepdims=True) + 1e-8
demand_norm    = demand_matrix    / demand_max
projected_norm = projected_demand / demand_max

print(f"✓ Demand normalised (0–1)")

✓ Demand matrix: (1488, 263)
  Total pickups : 2,843,292
  Max/bin/zone  : 330
  Sparsity      : 71.9% zero
✓ Travel matrix: (263, 263)
  Travel range (min) : 0.0 – 118.0
  Travel bins range  : 1 – 4
✓ Revenue matrix: (263, 263)
  Revenue range  : $0.01 – $580.00
✓ Projected demand: (1488, 263)
✓ Demand normalised (0–1)


## Features

### TIme features

In [8]:
# ============================================================
# CELL 8 — Build time features & save all
# ============================================================

# Redefine T_TOTAL (defensive)
T_TOTAL = 31 * BINS_PER_DAY

# ---- TIME FEATURES [T, 3] ----
bins = np.arange(T_TOTAL, dtype=np.float32)
hour_of_day = (bins // (60 / BIN_MINUTES)) % 24
day_of_week = ((bins // BINS_PER_DAY) + 6) % 7

sin_hour = np.sin(2 * np.pi * hour_of_day / 24).astype(np.float32)
cos_hour = np.cos(2 * np.pi * hour_of_day / 24).astype(np.float32)
cos_dow  = np.cos(2 * np.pi * day_of_week  /  7).astype(np.float32)

time_features = np.stack([sin_hour, cos_hour, cos_dow], axis=1)

print(f"✓ Time features: {time_features.shape}")
print(f"  sin_hour: {sin_hour.min():.3f} – {sin_hour.max():.3f}")
print(f"  cos_hour: {cos_hour.min():.3f} – {cos_hour.max():.3f}")
print(f"  cos_dow : {cos_dow.min():.3f}  – {cos_dow.max():.3f}")

# ---- SAVE ALL ----
np.save(f"{DATA_DIR}/demand_matrix.npy",    demand_matrix)
np.save(f"{DATA_DIR}/demand_norm.npy",      demand_norm)
np.save(f"{DATA_DIR}/projected_norm.npy",   projected_norm)
np.save(f"{DATA_DIR}/travel_bins.npy",      travel_bins)
np.save(f"{DATA_DIR}/avg_revenue.npy",      avg_revenue)
np.save(f"{DATA_DIR}/time_features.npy",    time_features)

print("\n✓ All tensors saved:")
for fname in ["demand_matrix", "demand_norm", "projected_norm",
              "travel_bins", "avg_revenue", "time_features"]:
    arr = np.load(f"{DATA_DIR}/{fname}.npy")
    print(f"  {fname}.npy → {arr.shape}  dtype={arr.dtype}")

✓ Time features: (1488, 3)
  sin_hour: -1.000 – 1.000
  cos_hour: -1.000 – 1.000
  cos_dow : -0.901  – 1.000

✓ All tensors saved:
  demand_matrix.npy → (1488, 263)  dtype=float32
  demand_norm.npy → (1488, 263)  dtype=float32
  projected_norm.npy → (1488, 263)  dtype=float32
  travel_bins.npy → (263, 263)  dtype=int32
  avg_revenue.npy → (263, 263)  dtype=float32
  time_features.npy → (1488, 3)  dtype=float32


In [9]:
# ============================================================
# CELL 9 — Load shapefile and build zone adjacency (with fallback)
# ============================================================
import geopandas as gpd
import os

print("Checking for NYC Taxi Zone shapefile ...")

shapefile_path = f"{ZONE_DIR}/taxi_zones.shp"

if os.path.exists(shapefile_path):
    print(f"✓ Shapefile found at {shapefile_path}")
    
    try:
        gdf = gpd.read_file(shapefile_path)
        print(f"✓ Loaded shapefile: {len(gdf)} zones")
        print(f"  Columns: {list(gdf.columns)}")

        # Filter to valid zones only (1–263)
        gdf_valid = gdf[gdf["LocationID"].between(1, N_ZONES)].copy()
        print(f"  Valid zones: {len(gdf_valid)}")

        gdf_valid = gdf_valid.sort_values("LocationID").reset_index(drop=True)

        # Build adjacency via polygon touches/overlaps
        print("\nComputing polygon adjacency (this may take ~30 sec) ...")

        n_valid = len(gdf_valid)
        adjacency_dense = np.zeros((N_ZONES, N_ZONES), dtype=np.uint8)

        for idx_i, (_, row_i) in enumerate(gdf_valid.iterrows()):
            zone_i = int(row_i["LocationID"])
            geom_i = row_i["geometry"]
            
            for idx_j, (_, row_j) in enumerate(gdf_valid.iterrows()):
                zone_j = int(row_j["LocationID"])
                geom_j = row_j["geometry"]
                
                if geom_i.touches(geom_j) or geom_i.intersects(geom_j):
                    i_0 = zone_i - 1
                    j_0 = zone_j - 1
                    adjacency_dense[i_0, j_0] = 1
            
            if (idx_i + 1) % 50 == 0:
                print(f"  Processed {idx_i + 1}/{n_valid} zones")

        print("✓ Adjacency matrix built from shapefile")
        np.fill_diagonal(adjacency_dense, 0)
        
    except Exception as e:
        print(f"✗ Error reading shapefile: {e}")
        print("  Falling back to trip-based adjacency...")
        raise

else:
    print(f"✗ Shapefile not found at {shapefile_path}")
    print(f"\nDirectory contents:")
    if os.path.exists(ZONE_DIR):
        for f in os.listdir(ZONE_DIR)[:20]:
            print(f"  {f}")
    else:
        print(f"  Directory {ZONE_DIR} does not exist!")
    
    print("\nFallback: Building adjacency from trip OD pairs...")
    print("(Zones are adjacent if there are direct trips between them)")
    
    # Build from OD pairs in trips_df
    adjacency_dense = np.zeros((N_ZONES, N_ZONES), dtype=np.uint8)
    
    od_pairs = (
        trips_df
        .select(["pu_zone", "do_zone"])
        .unique()
    )
    
    for row in od_pairs.iter_rows(named=True):
        pu = int(row["pu_zone"])
        do = int(row["do_zone"])
        if 1 <= pu <= N_ZONES and 1 <= do <= N_ZONES:
            i_0 = pu - 1
            j_0 = do - 1
            adjacency_dense[i_0, j_0] = 1
    
    np.fill_diagonal(adjacency_dense, 0)
    print("✓ Adjacency matrix built from OD pairs")

# Print stats
neighbor_counts = adjacency_dense.sum(axis=1)
print(f"\nAdjacency stats:")
print(f"  Edges (directed): {adjacency_dense.sum():.0f}")
print(f"  Neighbors per zone: {neighbor_counts.min():.0f} – {neighbor_counts.max():.0f}")
print(f"  Mean neighbors     : {neighbor_counts.mean():.1f}")
print(f"  Zones with 0 neighbors: {(neighbor_counts == 0).sum()}")
print(f"  Matrix shape: {adjacency_dense.shape}")

Checking for NYC Taxi Zone shapefile ...
✗ Shapefile not found at /content/tlc_data/taxi_zones/taxi_zones.shp

Directory contents:
  taxi_zones

Fallback: Building adjacency from trip OD pairs...
(Zones are adjacent if there are direct trips between them)
✓ Adjacency matrix built from OD pairs

Adjacency stats:
  Edges (directed): 20977
  Neighbors per zone: 0 – 257
  Mean neighbors     : 79.8
  Zones with 0 neighbors: 11
  Matrix shape: (263, 263)


In [10]:
# ============================================================
# CELL 10 — Convert adjacency to padded neighbor lists
# ============================================================

print("Building padded neighbor lists ...")

# Build neighbor lists
neighbor_lists = []
for i in range(N_ZONES):
    neighbors = np.where(adjacency_dense[i, :] == 1)[0]
    neighbor_lists.append(neighbors)

# Pad to MAX_DEGREE
neighbor_lists_padded = np.full((N_ZONES, MAX_DEGREE), -1, dtype=np.int32)

for i in range(N_ZONES):
    neighbors = neighbor_lists[i]
    n_neighbors = len(neighbors)
    
    if n_neighbors > MAX_DEGREE:
        neighbors = np.random.choice(neighbors, size=MAX_DEGREE, replace=False)
    
    neighbor_lists_padded[i, :n_neighbors] = neighbors

print(f"✓ Padded neighbor lists: {neighbor_lists_padded.shape}")
print(f"  Padding index (no neighbor): -1")
print(f"  Max neighbors per zone: {MAX_DEGREE}")

# Verify non-negative entries
print(f"  Min value: {neighbor_lists_padded.min()}")
print(f"  Max value: {neighbor_lists_padded.max()}")

Building padded neighbor lists ...
✓ Padded neighbor lists: (263, 10)
  Padding index (no neighbor): -1
  Max neighbors per zone: 10
  Min value: -1
  Max value: 262


In [11]:
# ============================================================
# CELL 11 — Compute k-hop neighborhoods
# ============================================================

print("Computing k-hop neighborhoods ...")

def compute_k_hop(adj_dense: np.ndarray, k: int, n_zones: int) -> np.ndarray:
    """Compute k-hop adjacency."""
    adj = adj_dense.copy().astype(np.float32)
    adj_k = np.eye(n_zones, dtype=np.float32)
    
    for _ in range(k):
        adj_k = np.minimum(adj_k @ adj + np.eye(n_zones), 1.0)
    
    return adj_k

adj_1hop = compute_k_hop(adjacency_dense, k=1, n_zones=N_ZONES)
adj_2hop = compute_k_hop(adjacency_dense, k=2, n_zones=N_ZONES)

print(f"✓ K-hop adjacencies computed:")
print(f"  1-hop edges: {(adj_1hop > 0).sum() // 2:.0f}")
print(f"  2-hop edges: {(adj_2hop > 0).sum() // 2:.0f}")

Computing k-hop neighborhoods ...
✓ K-hop adjacencies computed:
  1-hop edges: 10620
  2-hop edges: 31350


### Saving tensors

In [12]:
# ============================================================
# CELL 12 — Save adjacency matrices
# ============================================================

print("Saving adjacency matrices ...")

np.save(f"{DATA_DIR}/adjacency_dense.npy", adjacency_dense)
np.save(f"{DATA_DIR}/neighbor_lists_padded.npy", neighbor_lists_padded)
np.save(f"{DATA_DIR}/adj_1hop.npy", adj_1hop)
np.save(f"{DATA_DIR}/adj_2hop.npy", adj_2hop)

print("✓ Adjacency matrices saved:")
for fname in ["adjacency_dense", "neighbor_lists_padded", "adj_1hop", "adj_2hop"]:
    fpath = f"{DATA_DIR}/{fname}.npy"
    if os.path.exists(fpath):
        arr = np.load(fpath)
        size_mb = os.path.getsize(fpath) / 1e6
        print(f"  {fname}.npy → {arr.shape} dtype={arr.dtype} ({size_mb:.2f} MB)")
    else:
        print(f"  {fname}.npy → FAILED TO SAVE")

print("\n" + "="*60)
print("PHASE 1c SUMMARY — Zone Adjacency Graph")
print("="*60)
print(f"Zones: {N_ZONES}")
print(f"Edges: {adjacency_dense.sum() // 2:.0f}")
print(f"Avg degree: {adjacency_dense.sum() / N_ZONES:.1f}")
print(f"Neighbor lists padded to: {MAX_DEGREE}")
print("="*60)

Saving adjacency matrices ...
✓ Adjacency matrices saved:
  adjacency_dense.npy → (263, 263) dtype=uint8 (0.07 MB)
  neighbor_lists_padded.npy → (263, 10) dtype=int32 (0.01 MB)
  adj_1hop.npy → (263, 263) dtype=float64 (0.55 MB)
  adj_2hop.npy → (263, 263) dtype=float64 (0.55 MB)

PHASE 1c SUMMARY — Zone Adjacency Graph
Zones: 263
Edges: 10488
Avg degree: 79.8
Neighbor lists padded to: 10


### Verification and Loading

In [13]:
# ============================================================
# CELL 13 — Phase 1b Verification & Load/Rebuild
# ============================================================
import numpy as np
from pathlib import Path

# Define T_TOTAL globally
T_TOTAL = 31 * BINS_PER_DAY   # 1488

print("Checking Phase 1b outputs...")

# Try to load saved arrays; rebuild if missing
try:
    demand_matrix = np.load(f"{DATA_DIR}/demand_matrix.npy")
    demand_norm = np.load(f"{DATA_DIR}/demand_norm.npy")
    projected_norm = np.load(f"{DATA_DIR}/projected_norm.npy")
    travel_bins = np.load(f"{DATA_DIR}/travel_bins.npy")
    avg_revenue = np.load(f"{DATA_DIR}/avg_revenue.npy")
    time_features = np.load(f"{DATA_DIR}/time_features.npy")
    print("✓ All Phase 1b arrays loaded from disk")
except FileNotFoundError as e:
    print(f"⚠ Missing file: {e}")
    print("  Re-running Cell 6 → Cell 12 now...\n")
    
    # ===== REBUILD FROM SCRATCH =====
    
    # --- Cell 6 equivalent ---
    print("[Rebuild] Running DuckDB filter query...")
    QUERY = f"""
    SELECT
        tpep_pickup_datetime                        AS pickup_ts,
        tpep_dropoff_datetime                       AS dropoff_ts,
        PULocationID                                AS pu_zone,
        DOLocationID                                AS do_zone,
        CAST(fare_amount + tip_amount AS DOUBLE)    AS revenue,
        DATEDIFF('minute', tpep_pickup_datetime,
                           tpep_dropoff_datetime)   AS travel_minutes,
        FLOOR(
            DATEDIFF('minute',
                     TIMESTAMP '2023-01-01 00:00:00',
                     tpep_pickup_datetime)
            / {BIN_MINUTES}
        )::INTEGER AS time_bin
    FROM '{RAW_FILE}'
    WHERE
        tpep_pickup_datetime  >= TIMESTAMP '2023-01-01 00:00:00'
        AND tpep_pickup_datetime  <  TIMESTAMP '2023-02-01 00:00:00'
        AND tpep_dropoff_datetime >= tpep_pickup_datetime
        AND pu_zone BETWEEN 1 AND {N_ZONES}
        AND do_zone BETWEEN 1 AND {N_ZONES}
        AND fare_amount  > 0
        AND revenue      > 0
        AND travel_minutes BETWEEN 1 AND 120
        AND passenger_count >= 1
    """
    
    con = duckdb.connect()
    trips_df = pl.from_arrow(con.execute(QUERY).arrow())
    print(f"  ✓ Filtered trips: {len(trips_df):,}")
    
    # --- Cell 7 equivalent ---
    print("[Rebuild] Building demand matrix...")
    demand_raw = (
        trips_df
        .group_by(["time_bin", "pu_zone"])
        .agg(pl.len().alias("pickups"))
        .with_columns((pl.col("pu_zone") - 1).alias("zone_idx"))
    )
    
    demand_matrix = np.zeros((T_TOTAL, N_ZONES), dtype=np.float32)
    for row in demand_raw.iter_rows(named=True):
        t = int(row["time_bin"])
        z = int(row["zone_idx"])
        if 0 <= t < T_TOTAL and 0 <= z < N_ZONES:
            demand_matrix[t, z] = row["pickups"]
    print(f"  ✓ Demand shape: {demand_matrix.shape}")
    
    # --- Cell 8 equivalent ---
    print("[Rebuild] Building travel-time matrix...")
    travel_raw = (
        trips_df
        .group_by(["pu_zone", "do_zone"])
        .agg(pl.median("travel_minutes").alias("med_travel"))
        .with_columns([
            (pl.col("pu_zone") - 1).alias("pu_idx"),
            (pl.col("do_zone") - 1).alias("do_idx"),
        ])
    )
    
    travel_matrix = np.full((N_ZONES, N_ZONES), np.nan, dtype=np.float32)
    np.fill_diagonal(travel_matrix, 0.0)
    
    for row in travel_raw.iter_rows(named=True):
        i = int(row["pu_idx"])
        j = int(row["do_idx"])
        if 0 <= i < N_ZONES and 0 <= j < N_ZONES:
            travel_matrix[i, j] = row["med_travel"]
    
    for i in range(N_ZONES):
        for j in range(N_ZONES):
            if np.isnan(travel_matrix[i, j]) and not np.isnan(travel_matrix[j, i]):
                travel_matrix[i, j] = travel_matrix[j, i]
    
    global_med = np.nanmedian(travel_matrix)
    travel_matrix = np.where(np.isnan(travel_matrix), global_med, travel_matrix)
    travel_bins = np.ceil(travel_matrix / BIN_MINUTES).astype(np.int32)
    travel_bins = np.clip(travel_bins, 1, 10)
    print(f"  ✓ Travel bins shape: {travel_bins.shape}")
    
    # --- Cell 9 equivalent ---
    print("[Rebuild] Building projected demand...")
    H = DEMAND_FORECAST_HORIZON
    projected_demand = np.zeros((T_TOTAL, N_ZONES), dtype=np.float32)
    for h in range(1, H + 1):
        shifted = np.roll(demand_matrix, -h, axis=0)
        shifted[T_TOTAL - h:] = 0.0
        projected_demand += shifted
    projected_demand /= H
    
    demand_max = demand_matrix.max(axis=0, keepdims=True) + 1e-8
    demand_norm = demand_matrix / demand_max
    projected_norm = projected_demand / demand_max
    print(f"  ✓ Demand normalised")
    
    # --- Cell 10 equivalent ---
    print("[Rebuild] Building time features...")
    bins = np.arange(T_TOTAL, dtype=np.float32)
    hour_of_day = (bins // (60 / BIN_MINUTES)) % 24
    day_of_week = ((bins // BINS_PER_DAY) + 6) % 7
    sin_hour = np.sin(2 * np.pi * hour_of_day / 24).astype(np.float32)
    cos_hour = np.cos(2 * np.pi * hour_of_day / 24).astype(np.float32)
    cos_dow = np.cos(2 * np.pi * day_of_week / 7).astype(np.float32)
    time_features = np.stack([sin_hour, cos_hour, cos_dow], axis=1)
    print(f"  ✓ Time features shape: {time_features.shape}")
    
    # --- Cell 11 equivalent ---
    print("[Rebuild] Building revenue matrix...")
    revenue_raw = (
        trips_df
        .group_by(["pu_zone", "do_zone"])
        .agg(pl.mean("revenue").alias("avg_rev"))
        .with_columns([
            (pl.col("pu_zone") - 1).alias("pu_idx"),
            (pl.col("do_zone") - 1).alias("do_idx"),
        ])
    )
    
    avg_revenue = np.full((N_ZONES, N_ZONES), 0.0, dtype=np.float32)
    for row in revenue_raw.iter_rows(named=True):
        i = int(row["pu_idx"])
        j = int(row["do_idx"])
        if 0 <= i < N_ZONES and 0 <= j < N_ZONES:
            avg_revenue[i, j] = row["avg_rev"]
    
    global_avg_rev = avg_revenue[avg_revenue > 0].mean()
    avg_revenue = np.where(avg_revenue == 0, global_avg_rev, avg_revenue)
    print(f"  ✓ Revenue matrix shape: {avg_revenue.shape}")
    
    # --- Save ---
    np.save(f"{DATA_DIR}/demand_matrix.npy", demand_matrix)
    np.save(f"{DATA_DIR}/demand_norm.npy", demand_norm)
    np.save(f"{DATA_DIR}/projected_norm.npy", projected_norm)
    np.save(f"{DATA_DIR}/travel_bins.npy", travel_bins)
    np.save(f"{DATA_DIR}/avg_revenue.npy", avg_revenue)
    np.save(f"{DATA_DIR}/time_features.npy", time_features)
    print("\n✓ All arrays rebuilt and saved")

# --- Verify all variables in scope ---
print("\n✓✓ Phase 1b Complete. Variables in scope:")
print(f"  demand_matrix   : {demand_matrix.shape}")
print(f"  demand_norm     : {demand_norm.shape}")
print(f"  projected_norm  : {projected_norm.shape}")
print(f"  travel_bins     : {travel_bins.shape}")
print(f"  avg_revenue     : {avg_revenue.shape}")
print(f"  time_features   : {time_features.shape}")
print(f"  T_TOTAL         : {T_TOTAL}")
print(f"  N_ZONES         : {N_ZONES}")

Checking Phase 1b outputs...
✓ All Phase 1b arrays loaded from disk

✓✓ Phase 1b Complete. Variables in scope:
  demand_matrix   : (1488, 263)
  demand_norm     : (1488, 263)
  projected_norm  : (1488, 263)
  travel_bins     : (263, 263)
  avg_revenue     : (263, 263)
  time_features   : (1488, 3)
  T_TOTAL         : 1488
  N_ZONES         : 263


## RL Environment

### Taxifleetenv initialisation

In [14]:
# ============================================================
# CELL 14 — TaxiFleetEnv: Custom Gymnasium environment
# ============================================================

from gymnasium import Env, spaces
from gymnasium.utils import seeding

class TaxiFleetEnv(Env):
    """
    Multi-zone taxi fleet rebalancing environment.
    
    State:  [t, N, 7] where 7 = [idle, demand, proj_demand, in_transit, sin_h, cos_h, cos_dow]
    Action: [N, K_max] flow fractions to neighbors (softmax normalized)
    Reward: Cooperative signal (revenue - relocation - unmet demand)
    """
    
    metadata = {"render_modes": []}
    
    def __init__(
        self,
        demand_norm: np.ndarray,        # [T, N]
        projected_norm: np.ndarray,     # [T, N]
        travel_bins: np.ndarray,        # [N, N]
        avg_revenue: np.ndarray,        # [N, N]
        time_features: np.ndarray,      # [T, 3]
        adjacency: np.ndarray,          # [N, 263]
        neighbor_lists: np.ndarray,     # [N, K]
        n_zones: int = 263,
        n_drivers: int = 3000,
        n_bins_per_episode: int = 48,
        reward_weights: tuple = (1.0, 0.3, 0.5),
        device: torch.device = None,
    ):
        self.demand_norm = torch.FloatTensor(demand_norm).to(device)
        self.projected_norm = torch.FloatTensor(projected_norm).to(device)
        self.travel_bins = torch.LongTensor(travel_bins).to(device)
        self.avg_revenue = torch.FloatTensor(avg_revenue).to(device)
        self.time_features = torch.FloatTensor(time_features).to(device)
        self.adjacency = torch.FloatTensor(adjacency).to(device)
        self.neighbor_lists = torch.LongTensor(neighbor_lists)  # Keep on CPU for indexing
        
        self.n_zones = n_zones
        self.n_drivers = n_drivers
        self.n_bins_per_episode = n_bins_per_episode
        self.max_degree = neighbor_lists.shape[1]
        
        self.alpha, self.beta, self.gamma = reward_weights
        self.device = device if device else torch.device("cpu")
        
        # Observation space: [N, 7] features per zone
        self.observation_space = spaces.Box(
            low=-1.0, high=1.0,
            shape=(n_zones, 7),
            dtype=np.float32
        )
        
        # Action space: [N, K_max] flow fractions (bounded to [0, 1])
        self.action_space = spaces.Box(
            low=0.0, high=1.0,
            shape=(n_zones, self.max_degree),
            dtype=np.float32
        )
        
        # Running statistics for reward normalization (Welford)
        self.reward_mean = 0.0
        self.reward_var = 1.0
        self.reward_count = 0
        
        self.reset()
    
    def reset(self, seed=None, options=None):
        """Initialize episode."""
        super().reset(seed=seed)
        
        # Random start time (avoid last n_bins_per_episode to stay within bounds)
        max_start = len(self.demand_norm) - self.n_bins_per_episode - 1
        self.current_bin = self.np_random.integers(0, max(1, max_start))
        
        # Initialize driver distribution (proportional to mean demand per zone)
        mean_demand_per_zone = self.demand_norm.mean(dim=0)
        drivers_per_zone = (mean_demand_per_zone / mean_demand_per_zone.sum()) * self.n_drivers
        self.driver_counts = drivers_per_zone.clone()  # [N], float
        
        # In-transit drivers per zone (drivers currently on trips, not idle)
        self.in_transit = torch.zeros(self.n_zones, device=self.device)
        
        # Trip queue per zone (unmet demand)
        self.unmet_demand = torch.zeros(self.n_zones, device=self.device)
        
        self.steps_taken = 0
        
        return self._get_observation(), {}
    
    def _get_observation(self) -> np.ndarray:
        """Build state vector: [N, 7]"""
        t = self.current_bin
        
        # Fetch features for current bin
        demand_t = self.demand_norm[t]              # [N]
        projected_t = self.projected_norm[t]        # [N]
        time_feat_t = self.time_features[t]         # [3]
        
        # Idle drivers per zone (clipped to non-negative)
        idle_drivers = torch.clamp(self.driver_counts - self.in_transit, min=0.0)
        
        # Normalize idle drivers to [0, 1]
        max_idle = idle_drivers.max() + 1e-8
        idle_norm = idle_drivers / max_idle
        
        # Normalize in-transit to [0, 1]
        max_intransit = self.in_transit.max() + 1e-8
        intransit_norm = self.in_transit / max_intransit
        
        # Repeat time features for all zones
        time_feat_expanded = time_feat_t.unsqueeze(0).expand(self.n_zones, -1)  # [N, 3]
        
        # Concatenate: [idle, demand, projected, in_transit, sin_h, cos_h, cos_dow]
        obs = torch.cat([
            idle_norm.unsqueeze(1),
            demand_t.unsqueeze(1),
            projected_t.unsqueeze(1),
            intransit_norm.unsqueeze(1),
            time_feat_expanded
        ], dim=1)  # [N, 7]
        
        return obs.cpu().numpy().astype(np.float32)
    
    def step(self, action: np.ndarray) -> tuple:
        """
        Execute one environment step.
        
        action: [N, K_max] floats in [0, 1] (raw network output)
              Interpreted as logits for softmax → flow fractions
        """
        t = self.current_bin
        action_t = torch.FloatTensor(action).to(self.device)  # [N, K_max]
        
        # --- 1. RELOCATION: Apply action to move drivers to neighbors ---
        # Softmax over neighbor dimension to get flow fractions
        action_probs = torch.softmax(action_t, dim=1)  # [N, K_max]
        
        # Get current idle drivers
        idle_drivers = torch.clamp(self.driver_counts - self.in_transit, min=0.0)
        
        # For each zone, distribute idle drivers to neighbors
        relocation_cost = 0.0
        for i in range(self.n_zones):
            n_idle = idle_drivers[i].item()
            if n_idle < 0.1:
                continue  # Skip zones with no idle drivers
            
            # Get neighbors
            neighbors = self.neighbor_lists[i]  # [K_max], may contain -1
            valid_mask = neighbors >= 0
            valid_neighbors = neighbors[valid_mask]
            
            if len(valid_neighbors) == 0:
                continue  # Isolated zone
            
            # Get probabilities for valid neighbors
            probs = action_probs[i, valid_mask]
            probs = probs / (probs.sum() + 1e-8)  # Renormalize
            
            # Distribute drivers
            for j_idx, j in enumerate(valid_neighbors):
                flow = (probs[j_idx] * n_idle).item()
                self.driver_counts[i] -= flow
                self.driver_counts[j] += flow
                # Relocation cost = distance × drivers moved (travel time in bins)
                relocation_cost += self.travel_bins[i, j].float().item() * flow
        
        # Normalize relocation cost by bin size
        relocation_cost /= (self.n_drivers + 1e-8)
        
        # --- 2. PASSENGER MATCHING: Match idle drivers with current demand ---
        demand_t = self.demand_norm[t]  # [N]
        
        # Unmet demand accumulates
        self.unmet_demand += demand_t
        
        # Revenue from matched trips
        revenue_t = torch.zeros(self.n_zones, device=self.device)
        
        for i in range(self.n_zones):
            n_idle = torch.clamp(self.driver_counts[i] - self.in_transit[i], min=0.0)
            requests = self.unmet_demand[i]
            
            # Match min(idle, requests) drivers with passengers
            matched = torch.min(n_idle, requests)
            
            if matched > 0:
                # Revenue per zone
                rev_per_trip = self.avg_revenue[i, i]  # intra-zone trip (proxy)
                revenue_t[i] = matched * rev_per_trip
                
                # Update state
                self.driver_counts[i] -= matched
                self.in_transit[i] += matched
                self.unmet_demand[i] -= matched
        
        # --- 3. TRIP COMPLETION: Drivers finish trips and return ---
        # Assume all in-transit drivers complete (simplification)
        # In reality, this would depend on travel_bins distribution
        completed = self.in_transit.clone()
        self.in_transit = torch.zeros(self.n_zones, device=self.device)
        self.driver_counts += completed  # Drivers return to their current zone
        
        # --- 4. REWARD CALCULATION ---
        total_revenue = revenue_t.sum().item()
        unmet_penalty = self.unmet_demand.sum().item()
        
        # Normalize reward
        normalized_rev = total_revenue / (self.n_drivers + 1e-8)
        normalized_cost = relocation_cost / (self.n_drivers + 1e-8)
        normalized_unmet = unmet_penalty / (self.n_drivers + 1e-8)
        
        reward = (
            self.alpha * normalized_rev
            - self.beta * normalized_cost
            - self.gamma * normalized_unmet
        )
        
        # Clip reward
        reward = np.clip(reward, -REWARD_CLIP, REWARD_CLIP)
        
        # Update running reward statistics
        self.reward_count += 1
        delta = reward - self.reward_mean
        self.reward_mean += delta / self.reward_count
        delta2 = reward - self.reward_mean
        self.reward_var += delta * delta2
        
        # --- 5. ADVANCE TIME ---
        self.current_bin += 1
        self.steps_taken += 1
        
        # Episode terminates after n_bins_per_episode steps
        terminated = self.steps_taken >= self.n_bins_per_episode
        
        obs = self._get_observation()
        info = {
            "revenue": total_revenue,
            "relocation_cost": relocation_cost,
            "unmet_demand": unmet_penalty,
            "n_matched": revenue_t.sum().item(),
        }
        
        return obs, reward, terminated, False, info
    
    def seed(self, seed=None):
        self.np_random, seed = seeding.np_random(seed)
        return [seed]

### Instantiation

In [15]:
# ============================================================
# CELL 15 — Instantiate and test the environment
# ============================================================

print("Instantiating TaxiFleetEnv ...")

env = TaxiFleetEnv(
    demand_norm=demand_norm,                    # Already numpy
    projected_norm=projected_norm,              # Already numpy
    travel_bins=travel_bins,                    # Already numpy
    avg_revenue=avg_revenue,                    # Already numpy
    time_features=time_features,                # Already numpy
    adjacency=adjacency_dense,                  # Already numpy
    neighbor_lists=neighbor_lists_padded,       # Already numpy
    n_zones=N_ZONES,
    n_drivers=N_DRIVERS,
    n_bins_per_episode=ROLLOUT_STEPS,
    reward_weights=(ALPHA, BETA, GAMMA),
    device=DEVICE,
)

print(f"✓ Environment created")
print(f"  Observation space: {env.observation_space}")
print(f"  Action space: {env.action_space}")

# --- Test reset ---
obs, info = env.reset()
print(f"\n✓ Reset successful")
print(f"  Observation shape: {obs.shape}")
print(f"  Sample obs[0, :]: {obs[0, :]}")

# --- Test step ---
action = env.action_space.sample()  # Random action
obs_next, reward, terminated, truncated, info = env.step(action)

print(f"\n✓ Step successful")
print(f"  Reward: {reward:.4f}")
print(f"  Terminated: {terminated}")
print(f"  Info keys: {info.keys()}")
print(f"  Revenue: ${info['revenue']:.2f}")
print(f"  Relocation cost: {info['relocation_cost']:.2f}")
print(f"  Unmet demand: {info['unmet_demand']:.2f}")

# --- Rollout test (48 steps = 1 day) ---
print(f"\nRunning test rollout ({ROLLOUT_STEPS} steps) ...")
obs, info = env.reset()
rewards = []
for step in range(ROLLOUT_STEPS):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    rewards.append(reward)
    if terminated:
        break

print(f"✓ Rollout completed ({step+1} steps)")
print(f"  Cumulative reward: {sum(rewards):.4f}")
print(f"  Mean reward: {np.mean(rewards):.4f}")
print(f"  Min/Max reward: {min(rewards):.4f} / {max(rewards):.4f}")

print("\n" + "="*60)
print("PHASE 1d SUMMARY — Gymnasium Environment")
print("="*60)
print(f"Environment: TaxiFleetEnv")
print(f"Obs space: {env.observation_space}")
print(f"Action space: {env.action_space}")
print(f"Episode length: {ROLLOUT_STEPS} bins (24 hours)")
print(f"State features: idle, demand, proj_demand, in_transit, sin_h, cos_h, cos_dow")
print("="*60)

Instantiating TaxiFleetEnv ...
✓ Environment created
  Observation space: Box(-1.0, 1.0, (263, 7), float32)
  Action space: Box(0.0, 1.0, (263, 10), float32)

✓ Reset successful
  Observation shape: (263, 7)
  Sample obs[0, :]: [0.05359926 0.         0.         0.         0.         1.
 1.        ]

✓ Step successful
  Reward: 0.0457
  Terminated: False
  Info keys: dict_keys(['revenue', 'relocation_cost', 'unmet_demand', 'n_matched'])
  Revenue: $137.38
  Relocation cost: 1.41
  Unmet demand: 0.00

Running test rollout (48 steps) ...
✓ Rollout completed (48 steps)
  Cumulative reward: 5.1966
  Mean reward: 0.1083
  Min/Max reward: 0.0023 / 0.2155

PHASE 1d SUMMARY — Gymnasium Environment
Environment: TaxiFleetEnv
Obs space: Box(-1.0, 1.0, (263, 7), float32)
Action space: Box(0.0, 1.0, (263, 10), float32)
Episode length: 48 bins (24 hours)
State features: idle, demand, proj_demand, in_transit, sin_h, cos_h, cos_dow


### GCN implementation

In [16]:
# ============================================================
# CELL 16 — Sparse GCN implementation (manual, no torch_geometric)
# ============================================================

class SparseGCN(torch.nn.Module):
    """
    Single-layer sparse graph convolutional network.
    Efficiently aggregates node features with sparse adjacency.
    """
    
    def __init__(self, in_dim: int, out_dim: int, device: torch.device = None):
        super().__init__()
        self.in_dim = in_dim
        self.out_dim = out_dim
        self.device = device if device else torch.device("cpu")
        
        # Weight matrix: [in_dim, out_dim]
        self.weight = torch.nn.Parameter(
            torch.randn(in_dim, out_dim, device=self.device) * 0.01
        )
        
        # Bias
        self.bias = torch.nn.Parameter(
            torch.zeros(out_dim, device=self.device)
        )
        
    def forward(self, x: torch.Tensor, adj: torch.Tensor) -> torch.Tensor:
        """
        Aggregate and transform node features.
        
        Args:
            x: [N, in_dim] node features
            adj: [N, N] adjacency matrix (sparse or dense)
        
        Returns:
            out: [N, out_dim] aggregated features
        """
        # Linear transformation
        x_transformed = torch.matmul(x, self.weight)  # [N, out_dim]
        
        # Graph convolution: A @ X_transformed
        # If adj is sparse, use sparse-dense matmul
        if adj.is_sparse:
            out = torch.sparse.mm(adj, x_transformed)  # [N, out_dim]
        else:
            out = torch.matmul(adj, x_transformed)  # [N, out_dim]
        
        # Add bias
        out = out + self.bias
        
        return out


class GCNActor(torch.nn.Module):
    """
    Multi-agent actor policy using GCN message passing.
    
    Input:  state [N, 7]
    Output: action logits [N, K_max]
    """
    
    def __init__(
        self,
        n_zones: int,
        state_dim: int,
        max_degree: int,
        hidden_dim: int = 64,
        device: torch.device = None,
    ):
        super().__init__()
        self.n_zones = n_zones
        self.state_dim = state_dim
        self.max_degree = max_degree
        self.device = device if device else torch.device("cpu")
        
        # Graph convolution layers
        self.gcn1 = SparseGCN(state_dim, hidden_dim, device=self.device)
        self.gcn2 = SparseGCN(hidden_dim, hidden_dim // 2, device=self.device)
        
        # MLP head to output action logits
        self.mlp = torch.nn.Sequential(
            torch.nn.Linear(hidden_dim // 2, hidden_dim // 2, device=self.device),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden_dim // 2, max_degree, device=self.device),
        )
        
    def forward(self, state: torch.Tensor, adj: torch.Tensor) -> torch.Tensor:
        """
        Compute action logits for all zones.
        
        Args:
            state: [N, 7]
            adj: [N, N] adjacency
        
        Returns:
            action_logits: [N, K_max]
        """
        # GCN layers with ReLU
        x = self.gcn1(state, adj)
        x = torch.relu(x)
        x = self.gcn2(x, adj)
        x = torch.relu(x)
        
        # MLP to action logits
        action_logits = self.mlp(x)  # [N, K_max]
        
        return action_logits


class CentralizedCritic(torch.nn.Module):
    """
    Centralized value function using GCN global pooling.
    
    Input:  state [N, 7]
    Output: value scalar
    """
    
    def __init__(
        self,
        n_zones: int,
        state_dim: int,
        hidden_dim: int = 64,
        device: torch.device = None,
    ):
        super().__init__()
        self.n_zones = n_zones
        self.state_dim = state_dim
        self.device = device if device else torch.device("cpu")
        
        # GCN layers (same as actor for consistency)
        self.gcn1 = SparseGCN(state_dim, hidden_dim, device=self.device)
        self.gcn2 = SparseGCN(hidden_dim, hidden_dim // 2, device=self.device)
        
        # Global pooling + MLP to value
        self.mlp = torch.nn.Sequential(
            torch.nn.Linear(hidden_dim // 2, hidden_dim // 2, device=self.device),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden_dim // 2, 1, device=self.device),
        )
    
    def forward(self, state: torch.Tensor, adj: torch.Tensor) -> torch.Tensor:
        """
        Compute global value estimate.
        
        Args:
            state: [N, 7]
            adj: [N, N] adjacency
        
        Returns:
            value: scalar (or [1] tensor)
        """
        # GCN layers
        x = self.gcn1(state, adj)
        x = torch.relu(x)
        x = self.gcn2(x, adj)
        x = torch.relu(x)
        
        # Global mean pooling
        x_pooled = x.mean(dim=0, keepdim=True)  # [1, hidden_dim//2]
        
        # MLP to value
        value = self.mlp(x_pooled)  # [1, 1]
        
        return value.squeeze(-1)  # scalar


print("✓ GCN actor and critic classes defined")

✓ GCN actor and critic classes defined


In [17]:
# ============================================================
# CELL 17 — Convert adjacency to sparse PyTorch tensor
# ============================================================

print("Converting adjacency to PyTorch sparse tensor ...")

# Convert to COO format (torch expects edge indices + values)
rows, cols = np.nonzero(adjacency_dense)
edge_indices = torch.LongTensor(np.stack([rows, cols]))  # [2, E]
edge_weights = torch.ones(len(rows), dtype=torch.float32)

# Create sparse tensor (COO format)
adj_sparse = torch.sparse_coo_tensor(
    indices=edge_indices,
    values=edge_weights,
    size=(N_ZONES, N_ZONES),
    device=DEVICE,
)

# Convert to CSR for more efficient SpMM operations
adj_sparse = adj_sparse.coalesce()  # Merge duplicate indices

print(f"✓ Adjacency sparse tensor created:")
print(f"  Shape: {adj_sparse.shape}")
print(f"  Non-zeros: {adj_sparse._nnz()}")
print(f"  Format: COO (coalesced)")

# Also keep as dense for fallback
adj_dense_torch = torch.FloatTensor(adjacency_dense).to(DEVICE)
print(f"  Dense version also on device: {adj_dense_torch.device}")

Converting adjacency to PyTorch sparse tensor ...
✓ Adjacency sparse tensor created:
  Shape: torch.Size([263, 263])
  Non-zeros: 20977
  Format: COO (coalesced)
  Dense version also on device: cuda:0


/tmp/ipykernel_1070/3410624329.py:13: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  adj_sparse = torch.sparse_coo_tensor(


### Testing forward pass

In [18]:
# ============================================================
# CELL 18 — Instantiate actor and critic, test forward pass
# ============================================================

print("Instantiating GCN actor and critic ...")

actor = GCNActor(
    n_zones=N_ZONES,
    state_dim=GCN_IN_DIM,
    max_degree=MAX_DEGREE,
    hidden_dim=GCN_HIDDEN,
    device=DEVICE,
)

critic = CentralizedCritic(
    n_zones=N_ZONES,
    state_dim=GCN_IN_DIM,
    hidden_dim=GCN_HIDDEN,
    device=DEVICE,
)

print(f"✓ Actor instantiated")
print(f"  Parameters: {sum(p.numel() for p in actor.parameters()):,}")

print(f"✓ Critic instantiated")
print(f"  Parameters: {sum(p.numel() for p in critic.parameters()):,}")

# --- Test forward pass ---
print(f"\nTesting forward pass ...")

# Get observation from environment
obs, _ = env.reset()
obs_t = torch.FloatTensor(obs).to(DEVICE)  # [N, 7]

# Forward pass through actor
action_logits = actor(obs_t, adj_dense_torch)
print(f"✓ Actor output: {action_logits.shape}")
print(f"  Min/Max logits: {action_logits.min():.3f} / {action_logits.max():.3f}")

# Forward pass through critic
value = critic(obs_t, adj_dense_torch)
print(f"✓ Critic output: {value.shape}")
print(f"  Value: {value.item():.4f}")

# Test softmax on actions
action_probs = torch.softmax(action_logits, dim=1)
print(f"✓ Action softmax: {action_probs.shape}")
print(f"  Min/Max probs: {action_probs.min():.4f} / {action_probs.max():.4f}")
print(f"  Sum per zone (should be ~1.0): {action_probs[0, :].sum():.4f}")

print("\n" + "="*60)
print("PHASE 2a SUMMARY — GCN Actor + Critic")
print("="*60)
print(f"Actor architecture:")
print(f"  Input: [263, 7] state")
print(f"  GCN: 7 → 64 → 32")
print(f"  Output: [263, 10] action logits")
print(f"  Parameters: {sum(p.numel() for p in actor.parameters()):,}")
print(f"")
print(f"Critic architecture:")
print(f"  Input: [263, 7] state")
print(f"  GCN: 7 → 64 → 32")
print(f"  Global pool + MLP")
print(f"  Output: scalar value")
print(f"  Parameters: {sum(p.numel() for p in critic.parameters()):,}")
print("="*60)

Instantiating GCN actor and critic ...
✓ Actor instantiated
  Parameters: 3,978
✓ Critic instantiated
  Parameters: 3,681

Testing forward pass ...
✓ Actor output: torch.Size([263, 10])
  Min/Max logits: -2.076 / 3.556
✓ Critic output: torch.Size([1])
  Value: -0.8646
✓ Action softmax: torch.Size([263, 10])
  Min/Max probs: 0.0029 / 0.7965
  Sum per zone (should be ~1.0): 1.0000

PHASE 2a SUMMARY — GCN Actor + Critic
Actor architecture:
  Input: [263, 7] state
  GCN: 7 → 64 → 32
  Output: [263, 10] action logits
  Parameters: 3,978

Critic architecture:
  Input: [263, 7] state
  GCN: 7 → 64 → 32
  Global pool + MLP
  Output: scalar value
  Parameters: 3,681


In [19]:
# ============================================================
# CELL 19 — Rollout buffer for trajectory storage
# ============================================================

class RolloutBuffer:
    """
    Stores trajectories for one episode rollout.
    Computes GAE (Generalized Advantage Estimation).
    """
    
    def __init__(
        self,
        n_zones: int,
        max_steps: int,
        state_dim: int,
        action_dim: int,
        gamma: float = 0.99,
        gae_lambda: float = 0.95,
        device: torch.device = None,
    ):
        self.n_zones = n_zones
        self.max_steps = max_steps
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.gamma = gamma
        self.gae_lambda = gae_lambda
        self.device = device if device else torch.device("cpu")
        
        # Storage (pre-allocate for efficiency)
        self.states = []       # list of [N, state_dim]
        self.actions = []      # list of [N, action_dim]
        self.rewards = []      # list of scalars
        self.values = []       # list of scalars (critic outputs)
        self.log_probs = []    # list of [N,] or scalars
        self.dones = []        # list of bools
        self.infos = []        # list of dicts
        
        self.step_count = 0
    
    def add_transition(
        self,
        state: np.ndarray,
        action: np.ndarray,
        reward: float,
        value: float,
        log_prob: torch.Tensor,
        done: bool,
        info: dict,
    ):
        """Add one transition to buffer."""
        self.states.append(state)
        self.actions.append(action)
        self.rewards.append(reward)
        self.values.append(value)
        
        # Log prob may be [N,] or scalar — store as-is
        if isinstance(log_prob, torch.Tensor):
            self.log_probs.append(log_prob.detach().cpu().numpy())
        else:
            self.log_probs.append(log_prob)
        
        self.dones.append(done)
        self.infos.append(info)
        self.step_count += 1
    
    def compute_returns_and_advantages(self, final_value: float = 0.0):
        """
        Compute GAE advantages and returns.
        
        Args:
            final_value: Value estimate at the end of the rollout
                        (0.0 if episode terminated, otherwise V(s_final))
        """
        n_steps = len(self.rewards)
        
        # Convert to tensors on device
        values = torch.tensor(self.values + [final_value], dtype=torch.float32, device=self.device)
        rewards = torch.tensor(self.rewards, dtype=torch.float32, device=self.device)
        dones = torch.tensor(self.dones, dtype=torch.float32, device=self.device)
        
        # Temporal difference targets
        deltas = rewards + self.gamma * values[1:] * (1.0 - dones) - values[:-1]
        
        # GAE computation (backward pass)
        advantages = torch.zeros(n_steps, dtype=torch.float32, device=self.device)
        gae = 0.0
        for t in reversed(range(n_steps)):
            gae = deltas[t] + self.gamma * self.gae_lambda * (1.0 - dones[t]) * gae
            advantages[t] = gae
        
        # Returns = advantages + values
        returns = advantages + values[:-1]
        
        return advantages, returns
    
    def get_batch(self, batch_size: int, final_value: float = 0.0):
        """
        Yield minibatches of transitions.
        """
        advantages, returns = self.compute_returns_and_advantages(final_value)
        
        # Normalize advantages
        adv_mean = advantages.mean()
        adv_std = advantages.std() + 1e-8
        advantages = (advantages - adv_mean) / adv_std
        
        n_steps = len(self.rewards)
        indices = np.random.permutation(n_steps)
        
        for start_idx in range(0, n_steps, batch_size):
            end_idx = min(start_idx + batch_size, n_steps)
            batch_indices = indices[start_idx:end_idx]
            
            batch_states = torch.tensor(
                np.array([self.states[i] for i in batch_indices]),
                dtype=torch.float32,
                device=self.device,
            )  # [B, N, state_dim]
            
            batch_actions = torch.tensor(
                np.array([self.actions[i] for i in batch_indices]),
                dtype=torch.float32,
                device=self.device,
            )  # [B, N, action_dim]
            
            batch_log_probs = torch.tensor(
                np.array([self.log_probs[i] for i in batch_indices]),
                dtype=torch.float32,
                device=self.device,
            )  # [B, ...] shape depends on log_prob structure
            
            batch_advantages = advantages[batch_indices]  # [B]
            batch_returns = returns[batch_indices]  # [B]
            
            yield {
                "states": batch_states,
                "actions": batch_actions,
                "old_log_probs": batch_log_probs,
                "advantages": batch_advantages,
                "returns": batch_returns,
            }
    
    def reset(self):
        """Clear buffer for next episode."""
        self.states = []
        self.actions = []
        self.rewards = []
        self.values = []
        self.log_probs = []
        self.dones = []
        self.infos = []
        self.step_count = 0

print("✓ RolloutBuffer class defined")

✓ RolloutBuffer class defined


In [20]:
# ============================================================
# CELL 20 — PPO trainer with actor-critic updates
# ============================================================

class PPOTrainer:
    """
    Multi-Agent PPO trainer with centralized critic.
    """
    
    def __init__(
        self,
        actor: torch.nn.Module,
        critic: torch.nn.Module,
        adjacency: torch.Tensor,
        lr_actor: float = 3e-4,
        lr_critic: float = 1e-3,
        clip_eps: float = 0.2,
        entropy_coef: float = 0.01,
        value_loss_coef: float = 0.5,
        grad_clip: float = 0.5,
        device: torch.device = None,
    ):
        self.actor = actor
        self.critic = critic
        self.adjacency = adjacency
        self.device = device if device else torch.device("cpu")
        
        self.clip_eps = clip_eps
        self.entropy_coef = entropy_coef
        self.value_loss_coef = value_loss_coef
        self.grad_clip = grad_clip
        
        # Optimizers
        self.optimizer_actor = torch.optim.Adam(
            actor.parameters(), lr=lr_actor
        )
        self.optimizer_critic = torch.optim.Adam(
            critic.parameters(), lr=lr_critic
        )
        
        # Logging
        self.actor_loss_hist = []
        self.critic_loss_hist = []
        self.entropy_hist = []
    
    def compute_action_distribution(
        self,
        states: torch.Tensor,
    ) -> tuple:
        """
        Compute action distribution (Gaussian).
        
        Args:
            states: [B, N, state_dim] or [N, state_dim]
        
        Returns:
            mean: [B, N, action_dim] or [N, action_dim]
            std: scalar (shared across all agents)
        """
        # Handle batched or single state
        if states.dim() == 2:
            # Single state [N, state_dim]
            action_logits = self.actor(states, self.adjacency)
            mean = torch.softmax(action_logits, dim=1)  # [N, action_dim]
            std = torch.tensor(0.1, device=self.device)  # Fixed std for flow fractions
            return mean, std
        else:
            # Batched [B, N, state_dim]
            batch_size = states.shape[0]
            means = []
            for b in range(batch_size):
                action_logits = self.actor(states[b], self.adjacency)
                mean = torch.softmax(action_logits, dim=1)
                means.append(mean)
            mean = torch.stack(means)  # [B, N, action_dim]
            std = torch.tensor(0.1, device=self.device)
            return mean, std
    
    def compute_log_prob(
        self,
        actions: torch.Tensor,
        mean: torch.Tensor,
        std: torch.Tensor,
    ) -> torch.Tensor:
        """
        Log probability of actions under a Gaussian policy.
        
        Args:
            actions: [B, N, action_dim] or [N, action_dim]
            mean: [B, N, action_dim] or [N, action_dim]
            std: scalar
        
        Returns:
            log_prob: [B] or scalar (sum over all agents)
        """
        # Clamp std to avoid numerical issues
        std = torch.clamp(std, min=1e-6)
        
        # Gaussian log prob
        var = std ** 2
        log_prob_action = -0.5 * ((actions - mean) ** 2 / var + 2 * torch.log(std))
        
        # Sum over zones and action dims
        if actions.dim() == 2:
            # Single state: [N, action_dim]
            log_prob = log_prob_action.sum()
            return log_prob
        else:
            # Batched: [B, N, action_dim]
            log_prob = log_prob_action.sum(dim=(1, 2))  # [B]
            return log_prob
    
    def entropy(
        self,
        mean: torch.Tensor,
        std: torch.Tensor,
    ) -> torch.Tensor:
        """Entropy of Gaussian distribution."""
        return 0.5 * torch.log(2 * np.pi * np.e * std ** 2)
    
    def update(self, buffer: RolloutBuffer, ppo_epochs: int = 4, batch_size: int = 256):
        """
        PPO update: clipped surrogate loss + value loss + entropy.
        """
        # Get final value for GAE
        final_state = torch.FloatTensor(buffer.states[-1]).to(self.device)
        with torch.no_grad():
            final_value = self.critic(final_state, self.adjacency).item()
        
        # Compute advantages and returns
        advantages, returns = buffer.compute_returns_and_advantages(final_value)
        adv_mean = advantages.mean()
        adv_std = advantages.std() + 1e-8
        advantages = (advantages - adv_mean) / adv_std
        
        actor_losses = []
        critic_losses = []
        entropies = []
        
        for epoch in range(ppo_epochs):
            for batch in buffer.get_batch(batch_size, final_value):
                states = batch["states"]  # [B, N, state_dim]
                actions = batch["actions"]  # [B, N, action_dim]
                old_log_probs = batch["old_log_probs"]  # [B]
                batch_advantages = batch["advantages"]  # [B]
                batch_returns = batch["returns"]  # [B]
                
                # --- Actor update ---
                self.optimizer_actor.zero_grad()
                
                # Current policy
                mean, std = self.compute_action_distribution(states)  # [B, N, action_dim]
                new_log_probs = self.compute_log_prob(actions, mean, std)  # [B]
                
                # PPO clipped surrogate
                ratio = torch.exp(new_log_probs - old_log_probs)
                surr1 = ratio * batch_advantages
                surr2 = torch.clamp(ratio, 1 - self.clip_eps, 1 + self.clip_eps) * batch_advantages
                actor_loss = -torch.min(surr1, surr2).mean()
                
                # Entropy regularization
                ent = self.entropy(mean, std).mean()
                actor_loss = actor_loss - self.entropy_coef * ent
                
                actor_loss.backward()
                torch.nn.utils.clip_grad_norm_(self.actor.parameters(), self.grad_clip)
                self.optimizer_actor.step()
                
                actor_losses.append(actor_loss.item())
                entropies.append(ent.item())
                
                # --- Critic update ---
                self.optimizer_critic.zero_grad()
                
                values = torch.stack([
                    self.critic(states[b], self.adjacency) for b in range(states.shape[0])
                ])  # [B]
                
                critic_loss = torch.nn.functional.mse_loss(values.squeeze(-1), batch_returns)
                # values: [48, 1] -> [48]

                critic_loss = self.value_loss_coef * critic_loss
                
                critic_loss.backward()
                torch.nn.utils.clip_grad_norm_(self.critic.parameters(), self.grad_clip)
                self.optimizer_critic.step()
                
                critic_losses.append(critic_loss.item())
        
        # Log stats
        self.actor_loss_hist.append(np.mean(actor_losses))
        self.critic_loss_hist.append(np.mean(critic_losses))
        self.entropy_hist.append(np.mean(entropies))
        
        return {
            "actor_loss": np.mean(actor_losses),
            "critic_loss": np.mean(critic_losses),
            "entropy": np.mean(entropies),
        }

print("✓ PPOTrainer class defined")

✓ PPOTrainer class defined


### Instantiate buffer and trainer

In [21]:
# ============================================================
# CELL 21 — Instantiate buffer and trainer
# ============================================================

print("Instantiating PPO infrastructure ...")

# Rollout buffer
buffer = RolloutBuffer(
    n_zones=N_ZONES,
    max_steps=ROLLOUT_STEPS,
    state_dim=GCN_IN_DIM,
    action_dim=MAX_DEGREE,
    gamma=GAMMA_RL,
    gae_lambda=GAE_LAMBDA,
    device=DEVICE,
)

print(f"✓ Rollout buffer created")
print(f"  Max steps per episode: {ROLLOUT_STEPS}")
print(f"  State dim: {GCN_IN_DIM}")
print(f"  Action dim: {MAX_DEGREE}")

# Trainer
trainer = PPOTrainer(
    actor=actor,
    critic=critic,
    adjacency=adj_dense_torch,
    lr_actor=LR_ACTOR,
    lr_critic=LR_CRITIC,
    clip_eps=CLIP_EPS,
    entropy_coef=ENTROPY_COEF,
    value_loss_coef=VALUE_LOSS_COEF,
    grad_clip=GRAD_CLIP,
    device=DEVICE,
)

print(f"✓ PPO trainer created")
print(f"  Actor LR: {LR_ACTOR}")
print(f"  Critic LR: {LR_CRITIC}")
print(f"  Clip epsilon: {CLIP_EPS}")
print(f"  Entropy coef: {ENTROPY_COEF}")

print("\n" + "="*60)
print("PHASE 2b SUMMARY — PPO Infrastructure")
print("="*60)
print(f"Rollout buffer: stores trajectories for GAE")
print(f"PPO trainer: clipped surrogate + value loss + entropy")
print(f"PPO epochs per update: {PPO_EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print("="*60)

Instantiating PPO infrastructure ...
✓ Rollout buffer created
  Max steps per episode: 48
  State dim: 7
  Action dim: 10
✓ PPO trainer created
  Actor LR: 0.0003
  Critic LR: 0.001
  Clip epsilon: 0.2
  Entropy coef: 0.01

PHASE 2b SUMMARY — PPO Infrastructure
Rollout buffer: stores trajectories for GAE
PPO trainer: clipped surrogate + value loss + entropy
PPO epochs per update: 4
Batch size: 256


### Testing the rollout

In [22]:
# ============================================================
# CELL 22 — Test rollout + buffer + update cycle
# ============================================================

print("Testing PPO training loop (1 episode) ...")

# Reset environment and buffer
obs, _ = env.reset()
buffer.reset()

episode_rewards = []

# Collect one trajectory
for step in range(ROLLOUT_STEPS):
    obs_t = torch.FloatTensor(obs).to(DEVICE)
    
    # Policy forward
    with torch.no_grad():
        action_logits = actor(obs_t, adj_dense_torch)
        action_probs = torch.softmax(action_logits, dim=1)  # [N, action_dim]
        
        # Sample actions from the distribution
        actions = torch.clamp(action_probs, min=0.0, max=1.0).cpu().numpy()
        
        # Compute log prob
        action_t = torch.FloatTensor(actions).to(DEVICE)
        mean, std = trainer.compute_action_distribution(obs_t.unsqueeze(0))  # [1, N, action_dim]
        log_prob = trainer.compute_log_prob(action_t.unsqueeze(0), mean, std)
        
        # Value estimate
        value = critic(obs_t, adj_dense_torch).item()
    
    # Environment step
    obs_next, reward, terminated, truncated, info = env.step(actions)
    
    # Store in buffer
    buffer.add_transition(
        state=obs,
        action=actions,
        reward=reward,
        value=value,
        log_prob=log_prob,
        done=terminated or truncated,
        info=info,
    )
    
    episode_rewards.append(reward)
    obs = obs_next
    
    if terminated or truncated:
        break

print(f"✓ Trajectory collected: {len(buffer.rewards)} steps")
print(f"  Episode reward: {sum(episode_rewards):.4f}")
print(f"  Mean reward: {np.mean(episode_rewards):.4f}")

# PPO update
print(f"\nRunning {PPO_EPOCHS} PPO update epochs ...")
update_stats = trainer.update(buffer, ppo_epochs=PPO_EPOCHS, batch_size=BATCH_SIZE)

print(f"✓ PPO update complete:")
print(f"  Actor loss: {update_stats['actor_loss']:.4f}")
print(f"  Critic loss: {update_stats['critic_loss']:.4f}")
print(f"  Entropy: {update_stats['entropy']:.4f}")

print("\n" + "="*60)
print("PHASE 2b TEST — Single episode training")
print("="*60)
print(f"Rollout: {len(buffer.rewards)} steps collected")
print(f"Episode cumulative reward: {sum(episode_rewards):.4f}")
print(f"PPO update: {PPO_EPOCHS} epochs, batch size {BATCH_SIZE}")
print(f"Actor loss: {update_stats['actor_loss']:.4f}")
print(f"Critic loss: {update_stats['critic_loss']:.4f}")
print("="*60)

Testing PPO training loop (1 episode) ...
✓ Trajectory collected: 48 steps
  Episode reward: 6.0806
  Mean reward: 0.1267

Running 4 PPO update epochs ...
✓ PPO update complete:
  Actor loss: 0.0088
  Critic loss: 0.4629
  Entropy: -0.8836

PHASE 2b TEST — Single episode training
Rollout: 48 steps collected
Episode cumulative reward: 6.0806
PPO update: 4 epochs, batch size 256
Actor loss: 0.0088
Critic loss: 0.4629


### Training loop - 600 episodes

In [23]:
# ============================================================
# CELL 23 — Training loop with logging
# ============================================================

print("Starting MAPPO training ({} episodes) ...".format(N_EPISODES))
print("="*60)

# Training history
training_history = {
    "episode": [],
    "episode_reward": [],
    "actor_loss": [],
    "critic_loss": [],
    "entropy": [],
    "revenue": [],
    "relocation_cost": [],
    "unmet_demand": [],
    "n_matched": [],
}

try:
    for episode in range(N_EPISODES):
        # Reset environment and buffer
        obs, _ = env.reset()
        buffer.reset()
        
        episode_rewards = []
        episode_info = {
            "revenue": [],
            "relocation_cost": [],
            "unmet_demand": [],
            "n_matched": [],
        }
        
        # Collect trajectory for this episode
        for step in range(ROLLOUT_STEPS):
            obs_t = torch.FloatTensor(obs).to(DEVICE)
            
            # Policy forward (no grad)
            with torch.no_grad():
                action_logits = actor(obs_t, adj_dense_torch)
                action_probs = torch.softmax(action_logits, dim=1)
                actions = torch.clamp(action_probs, min=0.0, max=1.0).cpu().numpy()
                
                # Log prob
                action_t = torch.FloatTensor(actions).to(DEVICE)
                mean, std = trainer.compute_action_distribution(obs_t.unsqueeze(0))
                log_prob = trainer.compute_log_prob(action_t.unsqueeze(0), mean, std)
                
                # Value
                value = critic(obs_t, adj_dense_torch).item()
            
            # Environment step
            obs_next, reward, terminated, truncated, info = env.step(actions)
            
            # Store
            buffer.add_transition(
                state=obs,
                action=actions,
                reward=reward,
                value=value,
                log_prob=log_prob,
                done=terminated or truncated,
                info=info,
            )
            
            episode_rewards.append(reward)
            for key in episode_info.keys():
                episode_info[key].append(info.get(key, 0.0))
            
            obs = obs_next
            
            if terminated or truncated:
                break
        
        # PPO update
        update_stats = trainer.update(
            buffer,
            ppo_epochs=PPO_EPOCHS,
            batch_size=BATCH_SIZE,
        )
        
        # Log episode stats
        training_history["episode"].append(episode + 1)
        training_history["episode_reward"].append(sum(episode_rewards))
        training_history["actor_loss"].append(update_stats["actor_loss"])
        training_history["critic_loss"].append(update_stats["critic_loss"])
        training_history["entropy"].append(update_stats["entropy"])
        training_history["revenue"].append(np.mean(episode_info["revenue"]))
        training_history["relocation_cost"].append(np.mean(episode_info["relocation_cost"]))
        training_history["unmet_demand"].append(np.mean(episode_info["unmet_demand"]))
        training_history["n_matched"].append(np.mean(episode_info["n_matched"]))
        
        # Print progress every 50 episodes
        if (episode + 1) % 50 == 0:
            print(f"Episode {episode + 1:4d}/{N_EPISODES}  |  "
                  f"Reward: {sum(episode_rewards):7.4f}  |  "
                  f"Actor Loss: {update_stats['actor_loss']:7.4f}  |  "
                  f"Critic Loss: {update_stats['critic_loss']:7.4f}  |  "
                  f"Entropy: {update_stats['entropy']:6.4f}")

except KeyboardInterrupt:
    print("\n✗ Training interrupted by user")

print("\n" + "="*60)
print(f"✓ Training complete ({len(training_history['episode'])} episodes)")
print("="*60)

Starting MAPPO training (600 episodes) ...

✗ Training interrupted by user

✓ Training complete (2 episodes)


### Save trained model

In [ ]:
# ============================================================
# CELL 24 — Save trained model
# ============================================================

import json

# Create checkpoint dict
checkpoint = {
    "actor_state_dict": actor.state_dict(),
    "critic_state_dict": critic.state_dict(),
    "optimizer_actor_state_dict": trainer.optimizer_actor.state_dict(),
    "optimizer_critic_state_dict": trainer.optimizer_critic.state_dict(),
    "episode": len(training_history["episode"]),
    "training_history": training_history,
}

checkpoint_path = f"{DATA_DIR}/mappo_checkpoint.pt"
torch.save(checkpoint, checkpoint_path)

print(f"✓ Model checkpoint saved → {checkpoint_path}")
print(f"  Episodes trained: {checkpoint['episode']}")
print(f"  File size: {os.path.getsize(checkpoint_path) / 1e6:.1f} MB")

# Save training history as JSON for analysis
history_path = f"{DATA_DIR}/training_history.json"
with open(history_path, "w") as f:
    # Convert numpy values to Python floats for JSON serialization
    history_json = {
        k: [float(v) for v in v_list]
        for k, v_list in training_history.items()
    }
    json.dump(history_json, f, indent=2)

print(f"✓ Training history saved → {history_path}")

### Summary

In [ ]:
# ============================================================
# CELL 25 — Print training summary statistics
# ============================================================

import json

print("\n" + "="*60)
print("PHASE 2c SUMMARY — Training Results")
print("="*60)

# Load history
with open(f"{DATA_DIR}/training_history.json", "r") as f:
    history = json.load(f)

# Compute statistics over last 100 episodes
last_n = 100
last_episodes = history["episode"][-last_n:]
last_rewards = history["episode_reward"][-last_n:]
last_actor_loss = history["actor_loss"][-last_n:]
last_critic_loss = history["critic_loss"][-last_n:]
last_revenue = history["revenue"][-last_n:]
last_unmet = history["unmet_demand"][-last_n:]

print(f"\nTraining Configuration:")
print(f"  Total episodes: {len(history['episode'])}")
print(f"  Steps per episode: {ROLLOUT_STEPS}")
print(f"  PPO epochs: {PPO_EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")

print(f"\nFirst 10 Episodes (warmup):")
print(f"  Avg episode reward: {np.mean(history['episode_reward'][:10]):.4f}")
print(f"  Avg actor loss: {np.mean(history['actor_loss'][:10]):.4f}")
print(f"  Avg critic loss: {np.mean(history['critic_loss'][:10]):.4f}")

print(f"\nLast {last_n} Episodes (final):")
print(f"  Avg episode reward: {np.mean(last_rewards):.4f}")
print(f"  Min/Max reward: {min(last_rewards):.4f} / {max(last_rewards):.4f}")
print(f"  Avg actor loss: {np.mean(last_actor_loss):.4f}")
print(f"  Avg critic loss: {np.mean(last_critic_loss):.4f}")
print(f"  Avg revenue/step: ${np.mean(last_revenue):.2f}")
print(f"  Avg unmet demand/step: {np.mean(last_unmet):.2f}")

print(f"\nLearning Curves:")
print(f"  Episode reward trend:")
print(f"    Eps 1-50    → {np.mean(history['episode_reward'][0:50]):.4f}")
print(f"    Eps 200-250 → {np.mean(history['episode_reward'][199:250]):.4f}")
print(f"    Eps 500-550 → {np.mean(history['episode_reward'][499:550]):.4f}" if len(history['episode']) >= 550 else "    (not reached)")
print(f"    Final {last_n} → {np.mean(last_rewards):.4f}")

print(f"\nActor/Critic Loss:")
print(f"  Actor loss (final {last_n}): {np.mean(last_actor_loss):.6f}")
print(f"  Critic loss (final {last_n}): {np.mean(last_critic_loss):.6f}")

print("\n" + "="*60)

### Testing the trained policy

In [ ]:
# ============================================================
# CELL 26 — Test trained policy (greedy rollout)
# ============================================================

print("\nTesting trained policy (greedy deterministic) ...")
print("="*60)

# Disable gradient computation
actor.eval()
critic.eval()

test_obs, _ = env.reset()
test_rewards = []
test_info_agg = {
    "revenue": [],
    "relocation_cost": [],
    "unmet_demand": [],
    "n_matched": [],
}

print(f"\nRunning deterministic policy rollout ({ROLLOUT_STEPS} steps) ...")

for step in range(ROLLOUT_STEPS):
    obs_t = torch.FloatTensor(test_obs).to(DEVICE)
    
    with torch.no_grad():
        action_logits = actor(obs_t, adj_dense_torch)
        action_probs = torch.softmax(action_logits, dim=1)
        # Use mean (deterministic) instead of sampling
        actions = action_probs.cpu().numpy()
    
    test_obs, reward, terminated, truncated, info = env.step(actions)
    test_rewards.append(reward)
    
    for key in test_info_agg.keys():
        test_info_agg[key].append(info.get(key, 0.0))
    
    if terminated or truncated:
        break

actor.train()
critic.train()

print(f"✓ Test rollout complete ({len(test_rewards)} steps)")
print(f"\nTest Performance:")
print(f"  Cumulative reward: {sum(test_rewards):.4f}")
print(f"  Avg reward/step: {np.mean(test_rewards):.4f}")
print(f"  Total revenue: ${sum(test_info_agg['revenue']):.2f}")
print(f"  Total relocation cost: {sum(test_info_agg['relocation_cost']):.2f}")
print(f"  Total unmet demand: {sum(test_info_agg['unmet_demand']):.2f}")
print(f"  Total matched requests: {sum(test_info_agg['n_matched']):.0f}")

print("\n" + "="*60)
print("PHASE 2 COMPLETE — MAPPO Training Finished")
print("="*60)
print(f"✓ Actor trained: {sum(p.numel() for p in actor.parameters()):,} params")
print(f"✓ Critic trained: {sum(p.numel() for p in critic.parameters()):,} params")
print(f"✓ Episodes: {len(training_history['episode'])}")
print(f"✓ Checkpoint saved")
print("="*60)

In [24]:
# ============================================================
# CELL 27 — Fixed GCN Actor with learnable log_std
# ============================================================

class GCNActorFixed(torch.nn.Module):
    """
    Fixed actor with learnable log_std for proper exploration.
    """
    
    def __init__(
        self,
        n_zones: int,
        state_dim: int,
        max_degree: int,
        hidden_dim: int = 64,
        device: torch.device = None,
    ):
        super().__init__()
        self.n_zones = n_zones
        self.state_dim = state_dim
        self.max_degree = max_degree
        self.device = device if device else torch.device("cpu")
        
        # GCN layers
        self.gcn1 = SparseGCN(state_dim, hidden_dim, device=self.device)
        self.gcn2 = SparseGCN(hidden_dim, hidden_dim // 2, device=self.device)
        
        # MLP head
        self.mlp = torch.nn.Sequential(
            torch.nn.Linear(hidden_dim // 2, hidden_dim // 2, device=self.device),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden_dim // 2, max_degree, device=self.device),
        )
        
        # Learnable log_std — starts at 0.0 (std=1.0, high exploration)
        # Shape [1] shared across all zones and actions
        self.log_std = torch.nn.Parameter(
            torch.zeros(1, device=self.device)
        )
    
    def forward(self, state: torch.Tensor, adj: torch.Tensor) -> tuple:
        """
        Returns:
            mean: [N, max_degree] softmax action means
            std:  scalar, exp(log_std) clamped to [0.01, 1.0]
        """
        x = self.gcn1(state, adj)
        x = torch.relu(x)
        x = self.gcn2(x, adj)
        x = torch.relu(x)
        
        action_logits = self.mlp(x)
        mean = torch.softmax(action_logits, dim=1)  # [N, max_degree]
        
        # Clamp std to valid range
        std = torch.exp(self.log_std).clamp(0.01, 1.0)
        
        return mean, std


class PPOTrainerFixed(PPOTrainer):
    """
    Fixed PPO trainer using learnable std from GCNActorFixed.
    """
    
    def compute_action_distribution(self, states: torch.Tensor) -> tuple:
        """Use actor's own learnable std."""
        if states.dim() == 2:
            mean, std = self.actor(states, self.adjacency)
            return mean, std
        else:
            batch_size = states.shape[0]
            means = []
            stds = []
            for b in range(batch_size):
                mean, std = self.actor(states[b], self.adjacency)
                means.append(mean)
                stds.append(std)
            mean = torch.stack(means)   # [B, N, action_dim]
            std = stds[0]               # scalar (shared)
            return mean, std


print("✓ GCNActorFixed and PPOTrainerFixed defined")

✓ GCNActorFixed and PPOTrainerFixed defined


In [ ]:
# ============================================================
# CELL 28 — Re-instantiate with fixed actor and retrain
# ============================================================

# Fresh actor and critic
actor_fixed = GCNActorFixed(
    n_zones=N_ZONES,
    state_dim=GCN_IN_DIM,
    max_degree=MAX_DEGREE,
    hidden_dim=GCN_HIDDEN,
    device=DEVICE,
)

critic_fixed = CentralizedCritic(
    n_zones=N_ZONES,
    state_dim=GCN_IN_DIM,
    hidden_dim=GCN_HIDDEN,
    device=DEVICE,
)

# Fresh buffer
buffer_fixed = RolloutBuffer(
    n_zones=N_ZONES,
    max_steps=ROLLOUT_STEPS,
    state_dim=GCN_IN_DIM,
    action_dim=MAX_DEGREE,
    gamma=GAMMA_RL,
    gae_lambda=GAE_LAMBDA,
    device=DEVICE,
)

# Fixed trainer
trainer_fixed = PPOTrainerFixed(
    actor=actor_fixed,
    critic=critic_fixed,
    adjacency=adj_dense_torch,
    lr_actor=LR_ACTOR,
    lr_critic=LR_CRITIC,
    clip_eps=CLIP_EPS,
    entropy_coef=0.05,         # Higher entropy bonus to force exploration
    value_loss_coef=VALUE_LOSS_COEF,
    grad_clip=GRAD_CLIP,
    device=DEVICE,
)

print(f"✓ Fixed actor params: {sum(p.numel() for p in actor_fixed.parameters()):,}")
print(f"  of which log_std: 1 param")
print(f"✓ Fixed critic params: {sum(p.numel() for p in critic_fixed.parameters()):,}")
print(f"✓ Entropy coef raised to 0.05 for exploration")

# ---- Retrain ----
print(f"\nRetraining for {N_EPISODES} episodes with fixed actor ...")
print("="*60)

training_history_fixed = {
    "episode": [],
    "episode_reward": [],
    "actor_loss": [],
    "critic_loss": [],
    "entropy": [],
    "log_std": [],
    "revenue": [],
    "unmet_demand": [],
}

try:
    for episode in range(N_EPISODES):
        obs, _ = env.reset()
        buffer_fixed.reset()
        
        episode_rewards = []
        episode_info = {"revenue": [], "unmet_demand": []}
        
        for step in range(ROLLOUT_STEPS):
            obs_t = torch.FloatTensor(obs).to(DEVICE)
            
            with torch.no_grad():
                mean, std = actor_fixed(obs_t, adj_dense_torch)
                
                # Sample from Gaussian centered on mean
                noise = torch.randn_like(mean) * std
                actions = torch.clamp(mean + noise, 0.0, 1.0).cpu().numpy()
                
                # Log prob
                action_t = torch.FloatTensor(actions).to(DEVICE)
                var = std ** 2
                log_prob = (-0.5 * ((action_t - mean) ** 2 / var)
                            - torch.log(std)).sum()
                
                value = critic_fixed(obs_t, adj_dense_torch).item()
            
            obs_next, reward, terminated, truncated, info = env.step(actions)
            
            buffer_fixed.add_transition(
                state=obs,
                action=actions,
                reward=reward,
                value=value,
                log_prob=log_prob,
                done=terminated or truncated,
                info=info,
            )
            
            episode_rewards.append(reward)
            episode_info["revenue"].append(info.get("revenue", 0.0))
            episode_info["unmet_demand"].append(info.get("unmet_demand", 0.0))
            
            obs = obs_next
            if terminated or truncated:
                break
        
        update_stats = trainer_fixed.update(
            buffer_fixed,
            ppo_epochs=PPO_EPOCHS,
            batch_size=BATCH_SIZE,
        )
        
        # Log
        current_log_std = actor_fixed.log_std.item()
        training_history_fixed["episode"].append(episode + 1)
        training_history_fixed["episode_reward"].append(sum(episode_rewards))
        training_history_fixed["actor_loss"].append(update_stats["actor_loss"])
        training_history_fixed["critic_loss"].append(update_stats["critic_loss"])
        training_history_fixed["entropy"].append(update_stats["entropy"])
        training_history_fixed["log_std"].append(current_log_std)
        training_history_fixed["revenue"].append(np.mean(episode_info["revenue"]))
        training_history_fixed["unmet_demand"].append(np.mean(episode_info["unmet_demand"]))
        
        if (episode + 1) % 50 == 0:
            print(f"Episode {episode+1:4d}/{N_EPISODES}  |  "
                  f"Reward: {sum(episode_rewards):7.4f}  |  "
                  f"Actor Loss: {update_stats['actor_loss']:8.5f}  |  "
                  f"Critic Loss: {update_stats['critic_loss']:7.4f}  |  "
                  f"log_std: {current_log_std:6.3f}")

except KeyboardInterrupt:
    print("\n Training interrupted")

print("\n" + "="*60)
print(f"✓ Retraining complete ({len(training_history_fixed['episode'])} episodes)")
print("="*60)

✓ Fixed actor params: 3,979
  of which log_std: 1 param
✓ Fixed critic params: 3,681
✓ Entropy coef raised to 0.05 for exploration

Retraining for 600 episodes with fixed actor ...


In [ ]:
# ============================================================
# CELL 29 — Save fixed model + print learning summary
# ============================================================

# Save
checkpoint_fixed = {
    "actor_state_dict": actor_fixed.state_dict(),
    "critic_state_dict": critic_fixed.state_dict(),
    "training_history": training_history_fixed,
}
torch.save(checkpoint_fixed, f"{DATA_DIR}/mappo_fixed_checkpoint.pt")
print(f"✓ Fixed model saved → {DATA_DIR}/mappo_fixed_checkpoint.pt")

# Summary
print("\n" + "="*60)
print("RETRAIN SUMMARY")
print("="*60)

rewards = training_history_fixed["episode_reward"]
log_stds = training_history_fixed["log_std"]
actor_losses = training_history_fixed["actor_loss"]

print(f"\nReward trend:")
print(f"  Eps   1-50  → {np.mean(rewards[0:50]):.4f}")
print(f"  Eps 200-250 → {np.mean(rewards[199:249]):.4f}" if len(rewards) >= 250 else "")
print(f"  Eps 500-550 → {np.mean(rewards[499:549]):.4f}" if len(rewards) >= 550 else "")
print(f"  Final 100   → {np.mean(rewards[-100:]):.4f}")

print(f"\nActor learning (should vary now):")
print(f"  Actor loss Eps 1-50  → {np.mean(actor_losses[0:50]):.6f}")
print(f"  Actor loss final 100 → {np.mean(actor_losses[-100:]):.6f}")

print(f"\nlog_std evolution (exploration decay):")
print(f"  Episode 1   → {log_stds[0]:.4f}")
print(f"  Episode 100 → {log_stds[99]:.4f}"  if len(log_stds) >= 100 else "")
print(f"  Episode 300 → {log_stds[299]:.4f}" if len(log_stds) >= 300 else "")
print(f"  Episode 600 → {log_stds[-1]:.4f}")

print(f"\nRevenue trend:")
print(f"  Avg revenue/step Eps 1-50  → ${np.mean(training_history_fixed['revenue'][0:50]):.2f}")
print(f"  Avg revenue/step final 100 → ${np.mean(training_history_fixed['revenue'][-100:]):.2f}")
print("="*60)